# Model Testing Notebook
Train, infer and evaluate Mamba, XGBoost and VAR on a local CSV.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yaml
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

import sys
sys.path.append("../src")   # adjust if your src path differs

from data_handler import DataHandler
from model import Model


## Config

In [ ]:
with open("../config/analysis_config.yaml") as f:
    cfg = yaml.safe_load(f)

# Indices into cfg["target"]["system__cycle_time"]:
#   0 = mamba, 1 = xgboost_forecast, 2 = var_forecast
# MAMBA_CFG   = cfg["target"]["system__cycle_time"][0]
# XGBOOST_CFG = cfg["target"]["system__cycle_time"][1]
# VAR_CFG     = cfg["target"]["system__cycle_time"][2]
HEALTH_SCORE =  cfg["target"]["ced_machine__health_score"][0]

TARGET = "ced_machine__health_score"
TRAIN_SPLIT = 0.8   # fraction of rows used for training



{'method': 'isolation_forest',
 'save_name': 'ced_machine__health_score',
 'model_type': 'sklearn',
 'required_features': ['ced_maintenance__maint_between_glue_purge_delay',
  'ced_maintenance__maint_between_nozzle_clean_delay',
  'ced_maintenance__maint_glue_purging_time',
  'ced_maintenance__maint_move_to_safe_time',
  'ced_maintenance__maint_nozzle_cleaning_time',
  'ced_maintenance__maint_post_glue_purge_delay',
  'ced_maintenance__maint_post_nozzle_clean_delay',
  'ced_maintenance__maintenance_waiting_time',
  'ced_station__barcode_scanning_time',
  'ced_station__between_cavities_delay',
  'ced_station__dispensing_time',
  'ced_station__downstream_waiting_time',
  'ced_station__entry_stopper_eval_delay',
  'ced_station__entry_stopper_lowering_time',
  'ced_station__entry_stopper_raising_time',
  'ced_station__exit_stopper_lowering_time',
  'ced_station__exit_stopper_raising_time',
  'ced_station__inspection_time',
  'ced_station__movein_to_entry_stopper_up_delay',
  'ced_station__

## Load data and train/test split

In [4]:
full_df = pd.read_csv("/home/dhruvkumarjiguda/code/asm-predictive-maintenance/services/analysis/src/training_data.csv")

split_idx = int(len(full_df) * TRAIN_SPLIT)
train_df  = full_df.iloc[:split_idx].reset_index(drop=True)
test_df   = full_df.iloc[split_idx:].reset_index(drop=True)

print(f"Total rows : {len(full_df)}")
print(f"Train rows : {len(train_df)}")
print(f"Test rows  : {len(test_df)}")


Total rows : 670
Train rows : 536
Test rows  : 134


## Helper — init handler and model

In [5]:
def make_handler_model(method_config, df):
    """Assign df to a fresh handler and return (handler, model)."""
    handler = DataHandler(config=method_config, target_name=TARGET)
    model   = Model(
        data_handler=handler,
        model=method_config["method"],
        config=method_config,
        target_name=TARGET,
    )
    handler.df = df[method_config["required_features"]].copy()
    return handler, model


def run_inference_all_windows(handler, model):
    """
    Slide over every available window in handler.df and collect predictions
    alongside the ground-truth target value at the last row of each window.
    Returns (y_true, y_pred) as 1-D numpy arrays.
    """
    y_true, y_pred = [], []
    curr_ts = None

    while True:
        window = handler.fetch_next_window(curr_first_timestamp=curr_ts, for_training=False)
        if window is None:
            break

        pred  = model.real_time_inference(window)
        truth = window.iloc[-1][TARGET]

        y_pred.append(pred[-1])
        y_true.append(truth)

        curr_ts = window.iloc[0]["timestamp"]

    return np.array(y_true), np.array(y_pred)


In [6]:
def plot_results(y_true_train, y_pred_train, y_true_test, y_pred_test, model_name):
    fig = plt.figure(figsize=(14, 9))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.3)

    # ── Train: pred vs real ───────────────────────────────────────────────
    ax0 = fig.add_subplot(gs[0, 0])
    ax0.plot(y_true_train, label="Real",      color="#2196F3", linewidth=1.2)
    ax0.plot(y_pred_train, label="Predicted", color="#FF2222", linewidth=1.2, linestyle="--")
    ax0.set_title(f"{model_name} — Train: Predicted vs Real")
    ax0.set_xlabel("Window index"); ax0.set_ylabel("cycle_time (s)")
    ax0.legend(); ax0.grid(True, alpha=0.3)

    # ── Test: pred vs real ────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 1])
    ax1.plot(y_true_test, label="Real",      color="#2196F3", linewidth=1.2)
    ax1.plot(y_pred_test, label="Predicted", color="#FF2222", linewidth=1.2, linestyle="--")
    ax1.set_title(f"{model_name} — Test: Predicted vs Real")
    ax1.set_xlabel("Window index"); ax1.set_ylabel("cycle_time (s)")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    # ── Train residuals ───────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, 0])
    res_train = y_pred_train - y_true_train
    ax2.bar(range(len(res_train)), res_train, color="#9C27B0", alpha=0.5)

    mae_train_ma = pd.Series(np.abs(res_train)).rolling(window=5).mean()
    ax2.plot(mae_train_ma, color="#FF9800", linewidth=2, label="MAE (MA-5)")

    ax2.axhline(0, color="black", linewidth=0.8)
    ax2.set_title(f"{model_name} — Train Residuals")
    ax2.set_xlabel("Window index"); ax2.set_ylabel("Error")
    ax2.legend(); ax2.grid(True, alpha=0.3)

    # ── Test residuals ────────────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 1])
    res_test = y_pred_test - y_true_test
    ax3.bar(range(len(res_test)), res_test, color="#9C27B0", alpha=0.5)

    mae_test_ma = pd.Series(np.abs(res_test)).rolling(window=5).mean()
    ax3.plot(mae_test_ma, color="#FF9800", linewidth=2, label="MAE (MA-5)")

    ax3.axhline(0, color="black", linewidth=0.8)
    ax3.set_title(f"{model_name} — Test Residuals")
    ax3.set_xlabel("Window index"); ax3.set_ylabel("Error")
    ax3.legend(); ax3.grid(True, alpha=0.3)

    # ── Metrics table ─────────────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[2, :])
    ax4.axis("off")
    metrics = [
        ["Split", "RMSE", "MAE", "Max Error"],
        ["Train",
         f"{root_mean_squared_error(y_true_train, y_pred_train):.4f}",
         f"{mean_absolute_error(y_true_train, y_pred_train):.4f}",
         f"{np.max(np.abs(y_pred_train - y_true_train)):.4f}"],
        ["Test",
         f"{root_mean_squared_error(y_true_test, y_pred_test):.4f}",
         f"{mean_absolute_error(y_true_test, y_pred_test):.4f}",
         f"{np.max(np.abs(y_pred_test - y_true_test)):.4f}"],
    ]
    tbl = ax4.table(cellText=metrics[1:], colLabels=metrics[0],
                    loc="center", cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1, 2)
    ax4.set_title(f"{model_name} — Metrics", pad=12)

    plt.suptitle(model_name, fontsize=14, fontweight="bold", y=1.01)
    plt.suptitle(model_name, fontsize=14, fontweight="bold", y=1.01)

    fig.savefig(f"{model_name}.png", dpi=300, bbox_inches="tight")

    display(fig)
    plt.close(fig)

---
## Mamba

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────
mamba_handler_train, mamba_model = make_handler_model(MAMBA_CFG, train_df)

X_train, y_train, ts_train = mamba_handler_train.fetch_train_data()
print("Mamba train X:", X_train.shape, "y:", y_train.shape)

mamba_model.train(X_train, y_train, ts_train)


In [ ]:
# ── Inference on train set ─────────────────────────────────────────────────
mamba_handler_train_infer, _ = make_handler_model(MAMBA_CFG, train_df)
mamba_handler_train_infer.df = mamba_handler_train.df   # reuse same df

# Point to saved weights so real_time_inference loads the trained model
MAMBA_CFG["load_path"] = f"saved_models/{MAMBA_CFG['save_name']}/model.pt"
mamba_model_infer, _ = make_handler_model(MAMBA_CFG, train_df)   # fresh model wrapper

mamba_y_true_train, mamba_y_pred_train = run_inference_all_windows(
    mamba_handler_train_infer, mamba_model
)
print("Train windows:", len(mamba_y_true_train))

# ── Inference on test set ──────────────────────────────────────────────────
mamba_handler_test, _ = make_handler_model(MAMBA_CFG, test_df)
mamba_y_true_test, mamba_y_pred_test = run_inference_all_windows(
    mamba_handler_test, mamba_model
)
print("Test windows:", len(mamba_y_true_test))


In [ ]:
plot_results(mamba_y_true_train, mamba_y_pred_train,
             mamba_y_true_test,  mamba_y_pred_test,
             model_name="Mamba")


---
## XGBoost

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────
xgb_handler_train, xgb_model = make_handler_model(XGBOOST_CFG, train_df)

X_train, y_train, ts_train = xgb_handler_train.fetch_train_data()
print("XGBoost train X:", X_train.shape, "y:", y_train.shape)

xgb_model.train(X_train, y_train, ts_train)


In [ ]:
# ── Inference on train set ─────────────────────────────────────────────────
xgb_handler_train_infer, _ = make_handler_model(XGBOOST_CFG, train_df)
XGBOOST_CFG["load_path"] = f"saved_models/{XGBOOST_CFG['save_name']}/model.joblib"

xgb_y_true_train, xgb_y_pred_train = run_inference_all_windows(
    xgb_handler_train_infer, xgb_model
)
print("Train windows:", len(xgb_y_true_train))

# ── Inference on test set ──────────────────────────────────────────────────
xgb_handler_test, _ = make_handler_model(XGBOOST_CFG, test_df)
xgb_y_true_test, xgb_y_pred_test = run_inference_all_windows(
    xgb_handler_test, xgb_model
)
print("Test windows:", len(xgb_y_true_test))


In [ ]:
plot_results(xgb_y_true_train, xgb_y_pred_train,
             xgb_y_true_test,  xgb_y_pred_test,
             model_name="XGBoost")


---
## VAR
> VAR does not need a separate train step — it fits in one shot on the training data then infers immediately.

In [ ]:
with open("../config/analysis_config.yaml") as f:
    cfg = yaml.safe_load(f)

VAR_CFG = cfg["target"]["system__cycle_time"][2]
VAR_CFG["load_path"] = None  # no joblib — VAR is kept in memory

TARGET = "system__cycle_time"
TRAIN_SPLIT = 0.8

full_df   = pd.read_csv("/home/dhruvkumarjiguda/code/asm-predictive-maintenance/services/analysis/src/4thFeb_df_combined_formatted.csv")
# split_idx = int(len(full_df) * TRAIN_SPLIT)
train_df  = full_df.iloc[:200].reset_index(drop=True)
test_df   = full_df.iloc[200:300].reset_index(drop=True)

var_handler, var_model = make_handler_model(VAR_CFG, train_df)

# Get the lag order from arch config so we know how many rows to feed
VAR_LAG_ORDER = VAR_CFG["arch"].get("maxlags", 5)

# Bypass SklearnBackend.train() — it splits X and calls predict() on raw (N, F)
# slices which VAR can't handle. Instead fit directly on the full time series.
feature_cols = [c for c in VAR_CFG["required_features"] if c != "timestamp"]
train_series = train_df[feature_cols].values  # (T, F)
var_model.backend.model.fit(train_series)
print(f"VAR fit complete | lag_order={var_model.backend.model._lag_order} | features={train_series.shape[1]}")


In [ ]:
def run_var_inference_all_windows(handler, model, lag_order):
    """Like run_inference_all_windows but slices window to lag_order rows for VAR."""
    y_true, y_pred = [], []
    curr_ts = None

    while True:
        window = handler.fetch_next_window(curr_first_timestamp=curr_ts, for_training=False)
        if window is None:
            break

        # VAR only needs the last lag_order rows, not the full history_window
        var_window = window.iloc[-lag_order:].copy()

        pred  = model.real_time_inference(var_window)
        truth = window.iloc[-1][TARGET]

        y_pred.append(pred[-1])
        y_true.append(truth)

        curr_ts = window.iloc[0]["timestamp"]

    return np.array(y_true), np.array(y_pred)


# Train windows
var_handler_train, _ = make_handler_model(VAR_CFG, train_df)
var_handler_train.df  = var_handler.df  # reuse same df
var_y_true_train, var_y_pred_train = run_var_inference_all_windows(
    var_handler_train, var_model, VAR_LAG_ORDER
)
print("Train windows:", len(var_y_true_train))

# Test windows
var_handler_test, _ = make_handler_model(VAR_CFG, test_df)
var_y_true_test, var_y_pred_test = run_var_inference_all_windows(
    var_handler_test, var_model, VAR_LAG_ORDER
)
print("Test windows:", len(var_y_true_test))


In [ ]:
plot_results(var_y_true_train, var_y_pred_train,
             var_y_true_test,  var_y_pred_test,
             model_name="VAR")

In [7]:
# ── Train ─────────────────────────────────────────────────────────────────
health_score_train, model = make_handler_model(HEALTH_SCORE, train_df)

X_train, y_train, ts_train = health_score_train.fetch_train_data()
print("iso train X:", X_train.shape, "y:", y_train.shape)

model.train(X_train, y_train, ts_train)


Required features :  ['ced_maintenance__maint_between_glue_purge_delay', 'ced_maintenance__maint_between_nozzle_clean_delay', 'ced_maintenance__maint_glue_purging_time', 'ced_maintenance__maint_move_to_safe_time', 'ced_maintenance__maint_nozzle_cleaning_time', 'ced_maintenance__maint_post_glue_purge_delay', 'ced_maintenance__maint_post_nozzle_clean_delay', 'ced_maintenance__maintenance_waiting_time', 'ced_station__barcode_scanning_time', 'ced_station__between_cavities_delay', 'ced_station__dispensing_time', 'ced_station__downstream_waiting_time', 'ced_station__entry_stopper_eval_delay', 'ced_station__entry_stopper_lowering_time', 'ced_station__entry_stopper_raising_time', 'ced_station__exit_stopper_lowering_time', 'ced_station__exit_stopper_raising_time', 'ced_station__inspection_time', 'ced_station__movein_to_entry_stopper_up_delay', 'ced_station__pallet_clamping_time', 'ced_station__pallet_lifting_time', 'ced_station__pallet_lowering_time', 'ced_station__pallet_movein_time', 'ced_sta

AttributeError: 'NoneType' object has no attribute 'shape'

---
## Model comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, split, results in [
    (axes[0], "Train", [
        ("Mamba",   mamba_y_true_train, mamba_y_pred_train),
        ("XGBoost", xgb_y_true_train,   xgb_y_pred_train),
        ("VAR",     var_y_true_train,    var_y_pred_train),
    ]),
    (axes[1], "Test", [
        ("Mamba",   mamba_y_true_test, mamba_y_pred_test),
        ("XGBoost", xgb_y_true_test,   xgb_y_pred_test),
        ("VAR",     var_y_true_test,    var_y_pred_test),
    ]),
]:
    names  = [r[0] for r in results]
    rmses  = [root_mean_squared_error(r[1], r[2]) for r in results]
    maes   = [mean_absolute_error(r[1], r[2])     for r in results]

    x = np.arange(len(names))
    w = 0.35
    ax.bar(x - w/2, rmses, w, label="RMSE", color="#2196F3", alpha=0.8)
    ax.bar(x + w/2, maes,  w, label="MAE",  color="#FF5722", alpha=0.8)
    ax.set_xticks(x); ax.set_xticklabels(names)
    ax.set_title(f"{split} — RMSE vs MAE"); ax.legend(); ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("Model Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()
